오픈 AI의 API 펑션 콜링을 사용해 현재 시간을 알려주는 책복을 랭체인으로 개발

@tool 데코레이터를 사용하면 함수를 도구로 변환할 수 있다. 이 데코레이터는 함수를 랭체인에서 외부 도구로 등록하여 언어모델이 함수를 호출하고 사용할 수 있게 해준다. 

@tool 데코레이터는 랭체인이 함수의 기능을 언어모델이 활용할 수 있게 만드는 역할

In [ ]:
##1. 언어모델 선언하기 
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 잘 지냈어요! 당신은 어떻게 지내고 계신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 12, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9400314673', 'id': 'chatcmpl-EmybxJ1S0xTAqJgTXQP4Lre9D13Pt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a202b0-5657-72f6-947d-439f74aa76fd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 18, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [39]:
## 2. 랭체인에 시간을 파닥하는 함수 추가하기
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time

In [40]:
## 3. 힘수 get_current_time 을 랭체인으로 llm에 연결하기 
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [41]:
# 4. 사용자의 질문과 tools 사용하여 llm 답변 생성
from langchain_core.messages import SystemMessage

messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7f0cfe5367', 'id': 'chatcmpl-EmybxROUbN252ZtNQZYQmRKGHeg7z', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a202b0-5a73-7689-8459-e5c8fe9ee34c-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'A

In [42]:
# 5. 도구 실행
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]

    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

Asia/Seoul (Busan) 현재시각 2026-11-22 18:57:22 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 135, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7f0cfe5367', 'id': 'chatcmpl-EmybxROUbN252ZtNQZYQmRKGHeg7z', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a202b0-5a73-7689-8459-e5c8fe9ee34c-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 

In [43]:
##6. 함수실행결과를 문장으로 출력
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 2026년 11월 22일 18시 57분입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 192, 'total_tokens': 215, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7f0cfe5367', 'id': 'chatcmpl-EmybyljVkB9V7JeYfIenveCrVyw8c', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a202b0-5d6f-702a-8786-7cb0491a263f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 192, 'output_tokens': 23, 'total_tokens': 215, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [44]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("태국은 지금 몇시야?")])

AIMessage(content='현재 태국의 시간은 한국 시간보다 2시간 느립니다. 예를 들어, 한국에서 오후 3시라면 태국에서는 오후 1시가 됩니다. 하지만 정확한 시간을 확인하려면 현재 시간을 기준으로 조정해야 합니다. 직접 시계를 확인해 주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 15, 'total_tokens': 75, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fe5dc95ec4', 'id': 'chatcmpl-EmybzkjqxbdphAlIexJEcli5VtTUf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a202b0-602f-7477-a7c9-fb136bd3c67f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 60, 'total_tokens': 75, 'input_token_details': {'a

In [45]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [46]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [47]:

%pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [48]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [49]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period) #판다스의 데이터프레임으로 변환
    history_md = history.to_markdown() #데이터프레임을 표로 변환

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [50]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 283, 'total_tokens': 310, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e327275518', 'id': 'chatcmpl-Emyc1pKKopSV6AcvpT342cLtVBl6c', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a202b0-695f-7050-8ffd-a665b04eab62-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_m7blvCOonfA1rVIwrjyjnwhA', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 283, 'output_tokens': 27, 'total_tokens

In [51]:
%pip install tabulate

Note: you may need to restart the kernel to use updated packages.


In [52]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg.content)


{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |
|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|
| 2026-10-21 00:00:00-04:00 | 381.43 | 388.84 | 380.51 |  386.47 | 4.88758e+07 |           0 |              0 |
| 2026-10-22 00:00:00-04:00 | 374.29 | 388.86 | 373.71 |  374.58 | 5.16109e+07 |           0 |              0 |
| 2026-10-23 00:00:00-04:00 | 373.2  | 377.32 | 365.27 |  372.22 | 3.4763e+07  |           0 |              0 |
| 2026-10-26 00:00:00-04:00 | 383.83 | 397.27 | 383.73 |  391.62 | 5.30902e+07 |           0 |              0 |
| 2026-10-27 00:00:00-04:00 | 397.76 | 408.45 | 395.25 |  407.46 | 4.53332e+07 |           0 |              0 |
| 2026-10-28 00:00:00-04:00 | 410.16 | 410.85 | 397.18 |  404.22 | 3.55988e+07 |           0 |              0 |
| 2026-10-29 00:00:00-04:00 | 405.63 | 406.

In [53]:
llm_with_tools.invoke(messages)

AIMessage(content='한 달 전 테슬라(TSLA)의 주가는 2026년 10월 21일에 386.47 달러로 마감되었습니다. 현재 주가는 2026년 11월 20일에 345.82 달러로 마감되었습니다. \n\n따라서, 테슬라의 주가는 한 달 전에 비해 내렸습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 1702, 'total_tokens': 1782, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e327275518', 'id': 'chatcmpl-Emyc3LhChrnqSRAoAUuIbNrUwlAhr', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a202b0-7102-70b8-ba8f-d988a90fbd7e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1702, 'output_tokens': 80, 'total_tokens': 1782, 'input_t

##스트림방식으로 출력하기: LLM이 생성되는 동안 텍스트 조각(chunk)를 하나씩 받아 출력하는 방식
언어모델로만 스트림 출력하기

In [54]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end='|') 


|안|녕하세요|!| 잘| 지|내|고| 있습니다|.| 한국| 사회|의| 문제|점|에| 대해| 이야기|해|보|면| 여러| 가지|가| 있습니다|.| 대표|적인| 몇| 가지|를| 소개|할|게|요|.

|1|.| **|고|용| 문제|**|:| 청|년| 실|업|률|이| 높|고|,| 정|규|직| 일|자리| 부족|이| 심|각|한| 문제|로| 지|적|됩니다|.| 많은| 청|년|들이| 비|정|규|직|이나| 아|르|바|이트|로| 일|하고| 있으며|,| 안정|된| 직|장을| 찾|기가| 어려|운| 상황|입니다|.

|2|.| **|주|거| 문제|**|:| 특히| 대|도|시|의| 부|동|산| 가격| 상승|으로| 인해| 주|거| 불|안|이| 큰| 이|슈|입니다|.| 젊|은| 세|대|가| 집|을| 소|유|하기| 어려|운| 환경|이| 조|성|되고| 있으며|,| 임|대|료|도| 계속| 오|름|세|를| 보|이고| 있습니다|.

|3|.| **|학|벌| 중심| 사회|**|:| 한국| 사회|는| 여|전히| 학|벌|이| 중요한| 요소|로| 작|용|하고| 있어|,| 학|력|에| 따라| 직|장|이나| 사회|적| 지|위|가| 결정|되는| 경우|가| 많|습니다|.| 이|로| 인해| 불|평|등|이| 심|화|되고| 있습니다|.

|4|.| **|정|신| 건강| 문제|**|:| 빠|른| 사회| 변화|와| 경쟁|으로| 인|한| 스트|레스| 등|으로| 정신| 건강| 문제가| 증가|하고| 있습니다|.| 우|울|증|이나| 불|안|장|애|를| 호|소|하는| 사람들이| 많|아|지고| 있으며|,| 이에| 대한| 인|식|과| 지원|도| 부족|한| 편|입니다|.

|5|.| **|성|차|별| 및| 성|폭|력|**|:| 여|전히| 성|차|별|과| 성|폭|력| 문제가| 심|각|합니다|.| 여성|의| 사회|적| 위치|와| 권|리가| 개선|되고| 있는| 점|도| 있지만|,| 여|전히| 많은| 문제|들이| 존재|합니다|.

|이| 외|에도| 다양한| 사회|적| 문제|들이| 있으며|,| 이|들을| 해결|하기| 위해|서는| 정부|,| 기업

In [55]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end='') 


안녕하세요! 저는 잘 지내고 있습니다. 한국 사회는 다양한 문제점들을 안고 있습니다. 몇 가지 주요 문제를 살펴보면:

1. **청년 실업**: 한국에서는 청년층 실업률이 높은 편입니다. 많은 졸업생들이 좋은 일자리를 찾기 힘들어 불안정한 고용 상태에 놓이거나, 비정규직으로 일하고 있습니다.

2. **주거 문제**: 특히 대도시에서의 주택 가격 상승은 큰 문제입니다. 젊은 세대는 내 집 마련이 어렵고, 전세 가격도 급등하여 많은 사람들이 주거 불안을 겪고 있습니다.

3. **고령화 사회**: 인구 고령화가 진행됨에 따라 노인 복지와 관련한 문제들이 대두되고 있습니다. 노인 인구가 증가하면서 그에 따른 사회적 비용과 돌봄 문제에 대한 대책이 필요합니다.

4. **정신 건강 문제**: 스트레스와 경쟁 사회로 인한 정신 건강 문제가 심각합니다. 많은 사람들이 우울증, 불안장애와 같은 정신 건강 문제를 겪고 있지만, 사회적 낙인 때문에 치료를 받지 못하는 경우가 많습니다.

5. **성차별 및 성폭력**: 성차별과 성폭력 문제 역시 여전히 해결되지 않은 큰 사회적 이슈입니다. 성평등을 위한 노력이 지속되고 있지만, 여전히 많은 갈등과 저항이 존재합니다.

6. **교육 격차**: 사교육의 의존도와 교육의 질 차이가 사회적 불평등을 초래하고 있습니다. 상류층 자녀와 그렇지 않은 자녀 간의 교육 접근성이 크게 다릅니다.

이 외에도 다양한 사회적 문제들이 존재하지만, 이를 해결하기 위한 노력과 대화가 지속적으로 이루어져야 할 것입니다. 각 문제는 복합적이며, 사회 전반의 변화가 필요합니다.

도구를 추가했을 때 스트림 출력하기 

In [56]:
##1. 도구 호출 청크 모아서 하나로 만들기
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_5cc4zfi7KFPFvweT8eAkecp0', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_5cc4zfi7KFPFvweT8eAkecp0', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_5cc4zfi7KFPFvweT8eAkecp0', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_5cc4zfi7KFPFvweT8eAkecp0', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_5cc4zfi7KFPFvweT8eAkecp0', 'type': 'tool_c

In [57]:
gathered

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e327275518', 'service_tier': 'default'}, id='lc_run--01a202b8-0991-70cb-a68e-6ff6818879ac', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_5cc4zfi7KFPFvweT8eAkecp0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 23, 'total_tokens': 226, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_5cc4zfi7KFPFvweT8eAkecp0', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

In [58]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재시각 2026-11-22 19:06:17 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e327275518', 'service_tier': 'default'}, id='lc_run--01a202b8-0991-70cb-a68e-6ff6818879ac', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_5cc4zfi7KFPFvweT8eAkecp0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 203, 'output_tokens': 23, 'total_tokens': 226, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_5cc4zfi7KFPFvweT8eAkecp0', 'index': 0, 'ty

In [59]:
for c in llm_with_tools.stream(messages):
    print(c.content, end='|')

|부|산|은| 지금| |202|6|년| |11|월| |22|일| |19|시| |6|분|입니다|.||||

In [60]:
for c in llm_with_tools.stream(messages):
    print(c.content, end='')

부산은 지금 2026년 11월 22일 19시 06분입니다.

- invoke() : 답변이 완성될 때까지 기다렸다가 한 번에 반환
- stream() : 답변이 생성되는 대로 조금씩 반환